# Use Case 1 - Name/M1 --> Proposal/Teams

<ol>
    <li>Generic Data Import. <a href="#gen_data_import">Here.</a></li>
    <li>Method/Files Import. <a href="#method_import">Here.</a></li>
    <li>M1 - String Matching. <a href="#m1">Here.</a></li>
</ol>

## Generic Data Import <a id='gen_data_import'></a> 

In [1]:
import pandas as pd
import numpy as np
# Import list of researchers
og_researchers=pd.read_csv('../data/v1_input_files/v1_researchers.csv')

# Import a subset of the proposals, sorted by the YEAR they were sent
proposal_info=pd.read_csv('../data/v1_input_files/v1_proposal_links_title_synopsis.csv')
proposal_info.sort_values(["nsf_proposal_links_v1"], 
                    ascending=[False], 
                    inplace=True)
# proposal_info=proposal_info[:100]
proposal_info.pop("Unnamed: 0")
proposal_info.reset_index(drop=True, inplace=True)

In [2]:
linebreak = "\n\n----------------------------------------------------------------"

print(og_researchers.iloc[0], linebreak)
print(proposal_info.iloc[0])

Unnamed: 0                                                      0
names                                         Agostinelli, Forest
descriptions                                  Assistant Professor
titles                                                    Faculty
research        ['Artificial Intelligence, Deep Learning, Rein...
Name: 0, dtype: object 

----------------------------------------------------------------
nsf_proposal_links_v1    https://www.nsf.gov/pubs/2021/nsf21598/nsf2159...
title                                     Advanced Technological Education
synopsis                 With a focus on two-year Institutions of Highe...
Name: 0, dtype: object


In [3]:
# error case encountered at the end - so preventing it now by excluding this proposal

error_values = ['https://www.nsf.gov/funding/pgm_summ.jsp?pims_id=505073'] 

#drop rows that contain any value in the list
proposal_info = proposal_info[proposal_info.nsf_proposal_links_v1.isin(error_values) == False]
proposal_info.reset_index(drop=True, inplace=True)

***

## Methods/Files Import <a id='method_import'></a> 

In [4]:
import nlp_techniques
import M1

[nltk_data] Downloading package wordnet to /Users/tej/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/tej/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## M1 - String Matching <a id="m1"></a>

<b>Steps:</b>
<ol>
    <li>Extract and preprocess a researcher's own skills (extracted from their homepage in the faculty directory). <a href="#method_import_1">Here.</a></li>
    <li>Store all researchers' skills in a separate list. <a href="#method_import_2">Here.</a></li>
    <li>Extract necessary "skills" required for each proposal, with everyone as a whole having them. <a href="#method_import_3">Here.</a></li>
    <li>Create teams for each proposal call and for each researcher based on proposal/researcher string matches. <a href="#method_import_4">Here.</a></li>
    <li>Apply Ultra-Metric. <a href="#method_import_5">Here.</a></li>
    <li>Final exports to CSV. <a href="#method_import_6">Here.</a></li>    
</ol>

### Step 1 - Extract and preprocess a researcher's own skills (extracted from their homepage in the faculty directory). <a id='method_import_1'></a> 

In [5]:
import ast
import datetime

# Start time
print("Start time:\t", datetime.datetime.now())

m1_researcher_skills={}

# for each researcher
for i in range(len(og_researchers["research"])):
    # load into variables
    researcher=og_researchers["names"][i]
    interests=og_researchers["research"][i]
    if type(interests)==float:  # account for nan values
        interests="['research', 'general', 'computer', 'science', 'engineering']"
    
    # convert the string of interests into a list of interests
    interests=ast.literal_eval(interests)[0].split(", ")
    for j in range(len(interests)):
        interests[j]=nlp_techniques.preprocess(interests[j])
        
    while '' in interests:
        interests.remove('')
    
    # print(researcher,interests)   # E.g., Agostinelli, Forest ['artificial intelligence', 'deep learning', ...]

    # do it for n-grams of 2 as well
    n_gram_interests=[]
    for j in range(len(interests)):
        n_gram_interests.append(nlp_techniques.generate_N_grams(interests[j], ngram=2))
        
    n_gram_interests=[item for sublist in n_gram_interests for item in sublist]   # merge list of lists into a flat list
    
    # merge interests
    merged_interests=set(interests+n_gram_interests)
    while '' in merged_interests:
        merged_interests.remove('')    # remove null/empty strings
    
    # save info
    m1_researcher_skills[researcher]=merged_interests
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:18.516320
End time:	 2026-04-01 15:28:21.757671


In [6]:
# save directory
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers/data_uc1_m1/"

# export m1_researcher_skills
csv_m1_researcher_skills=[]
for i in m1_researcher_skills:
    csv_m1_researcher_skills.append([i, m1_researcher_skills[i]])

csv_m1_researcher_skills=pd.DataFrame(csv_m1_researcher_skills, columns = ['researcher_name', 'skills'])
csv_m1_researcher_skills.to_csv(save_dir+'m1_researcher_skills.csv', encoding='utf-8')
del csv_m1_researcher_skills

### Step 2 - Store all researchers' skills in a separate list. <a id='method_import_2'></a> 

**Why?**

Because a proposal individually would have a lot of "skills" extracted, including the irrelevant terms (or those that the researchers are **not** familiar with). If a proposal has N skills extracted from its title/synopsis, then each of the i-th skill would only count IF that i-th skill is already in m1_all_researcher_skills[] (below).

This way, we are condensing the number of searches, and making Ultra-Metric more readable.

In [7]:
# Start time
print("Start time:\t", datetime.datetime.now())

# set of all skills that researchers have
m1_all_researcher_skills=[]

for i in m1_researcher_skills:    # compile a list of all skills that researchers have
    for j in m1_researcher_skills[i]:
        if j not in m1_all_researcher_skills:
            m1_all_researcher_skills.append(j)

# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:21.783111
End time:	 2026-04-01 15:28:22.122952


In [8]:
m1_all_researcher_skills

['reinforcement learning',
 'bioinformatics',
 'artificial intelligence',
 'search',
 'deep learning',
 'ultra wide',
 'aluminum content',
 'senior scientist',
 'electronic photonic',
 'bandgap semiconductor',
 'boron nitride gallium oxide',
 'position senior',
 'inc ph texas tech universityresearch growth study ultra wide bandgap semiconductor including high aluminum content algan',
 'study ultra',
 'high aluminum',
 'texas tech',
 'wide bandgap',
 'content algan',
 'high power',
 'novel high',
 'boron nitride',
 'gallium oxide',
 'nitride gallium',
 'computer simulation',
 'previous position senior scientist',
 'power electronic',
 'tech universityresearch',
 'fabrication novel high power electronic photonic device',
 'computer simulation device',
 'sensor electronic',
 'fabrication novel',
 'photonic device',
 'electronic technology',
 'universityresearch growth',
 'inc ph',
 'growth study',
 'semiconductor including',
 'ph texas',
 'including high',
 'previous position',
 'simulati

In [9]:
# export m1_all_researcher_skills
csv_m1_all_researcher_skills=[]
for i in m1_all_researcher_skills:
    csv_m1_all_researcher_skills.append(i)

csv_m1_proposal_skills=pd.DataFrame(csv_m1_all_researcher_skills, columns = ['all_skills'])
csv_m1_proposal_skills.to_csv(save_dir+'m1_all_researcher_skills.csv', encoding='utf-8')
del csv_m1_all_researcher_skills

### Step 3 - Extract necessary "skills" required for each proposal, with everyone as a whole having them. <a id='method_import_3'></a> 

In [10]:
# Start time
print("Start time:\t", datetime.datetime.now())

m1_proposal_skills={}

# for each proposal
for i in range(len(proposal_info["nsf_proposal_links_v1"])):
    # extract respective title/synopsis
    title=proposal_info["title"][i]
    synopsis=proposal_info["synopsis"][i]
        
    # check if title is an empty field
    if type(title)==float:     # if so, then assign a general value to title
        title="general"
    
    # check if synopsis is an empty field
    if type(synopsis)==float:     # if so, then assign an empty value to synopsis
        synopsis=""
    
    # preprocess title/synopsis
    title=nlp_techniques.preprocess(title)
    synopsis=nlp_techniques.preprocess(synopsis)
        
    # keywords of title - just split the string and apply set()
    keywords=title.split(" ")+synopsis.split(" ")

    # n-gram keywords
    title_n=nlp_techniques.generate_N_grams(title, ngram=2)
    synopsis_n=nlp_techniques.generate_N_grams(synopsis, ngram=2)
    
    n_gram_keywords=set(title+synopsis)
    
    # merge
    all_keywords=set(keywords+title_n+synopsis_n)
    
    # if any of these keywords do not exist in m1_all_researcher_skills, remove them
    skills_to_be_removed=[]
    for j in all_keywords:
        if j not in m1_all_researcher_skills:
            skills_to_be_removed.append(j)
            
    for j in skills_to_be_removed:
        all_keywords.remove(j)        
    
    # in case of empty set
    if all_keywords==set():
        all_keywords=set(["general"])
        
    # add them to the dictionary mapping
    m1_proposal_skills[proposal_info["nsf_proposal_links_v1"][i]]=all_keywords
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:22.175942
End time:	 2026-04-01 15:28:29.767125


In [11]:
# export m1_proposal_skills
csv_m1_proposal_skills=[]
for i in m1_proposal_skills:
    csv_m1_proposal_skills.append([i, m1_proposal_skills[i]])

csv_m1_proposal_skills=pd.DataFrame(csv_m1_proposal_skills, columns = ['nsf_proposal_links_v1', 'skills'])
csv_m1_proposal_skills.to_csv(save_dir+'m1_proposal_skills.csv', encoding='utf-8')
del csv_m1_proposal_skills

### Step 4 - Create teams for each proposal call and for each researcher based on proposal/researcher string matches. <a id='method_import_4'></a> 

In [12]:
string_matching_threshold=0.5

In [13]:
# Start time
print("Start time:\t", datetime.datetime.now())

m1_teaming={}  # data={people:proposal/team}
m1_pseudo_researcher_skills={}

# for each proposal 
for i in range(len(proposal_info["nsf_proposal_links_v1"])):
    proposal=proposal_info["nsf_proposal_links_v1"][i]    
    m1_teaming[proposal]=[]
    
    # because we're using a threshold count, skills that a researcher never had may also get selected. We keep track of this in a separate list
    m1_pseudo_researcher_skills[proposal]={}    
    count=0
    # for each researcher, create random N teams`
    for j in range(0,len(og_researchers["names"])):       
        target_researcher=og_researchers["names"][j]
        
        # call m1 function - string_matching_ranking()
        if m1_pseudo_researcher_skills[proposal]=={}:
            ranking, pseudo_researcher_skills=M1.string_matching_ranking(m1_researcher_skills, m1_proposal_skills[proposal], {}, matching_threshold=string_matching_threshold)            
            m1_pseudo_researcher_skills[proposal]=pseudo_researcher_skills
        else: 
            ranking=M1.string_matching_ranking(m1_researcher_skills, m1_proposal_skills[proposal], pseudo_researcher_skills, matching_threshold=string_matching_threshold)
        
        # call m1 function - create_teams_for_each_person()
        num_of_teams=10
        teams=M1.create_teams_for_each_person(ranking, target_researcher, num_of_teams)
        
        # save teams in the researcher's profile
        m1_teaming[proposal_info["nsf_proposal_links_v1"][i]].append([target_researcher,teams])
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:29.789749
End time:	 2026-04-01 15:33:11.298115


In [14]:
# export m1_pseudo_researcher_skills
csv_m1_pseudo_researcher_skills=[]
for i in m1_pseudo_researcher_skills:
    for j in m1_pseudo_researcher_skills[i]:
        csv_m1_pseudo_researcher_skills.append([i,j,m1_pseudo_researcher_skills[i][j]])

csv_m1_pseudo_researcher_skills=pd.DataFrame(csv_m1_pseudo_researcher_skills, columns = ['nsf_proposal_links_v1', 'researcher', 'pseudo_skills'])
csv_m1_pseudo_researcher_skills.to_csv(save_dir+'m1_pseudo_researcher_skills.csv', encoding='utf-8')
del csv_m1_pseudo_researcher_skills

# export m1_teaming
csv_m1_teaming=[]
for i in m1_teaming:   # proposal
    for j in m1_teaming[i]:  # researcher
        for k in j[1]:         # list of teams
            csv_m1_teaming.append([i, j[0], k])

csv_m1_teaming=pd.DataFrame(csv_m1_teaming, columns = ['nsf_proposal_links_v1', 'researcher', 'team'])
csv_m1_teaming.to_csv(save_dir+'m1_teaming.csv', encoding='utf-8')
del csv_m1_teaming

### Step 5 - Apply Ultra-Metric. <a id='method_import_5'></a> 

<b>Variables:</b>
<ol>
    <li><i>m1_teaming{}</i> - {proposal_link: [[researcher1, [team1, team2, ...], [researcher2, [team1, team2,...]]</li>
    <li><i>m1_proposal_skills{}</i> - {proposal_link: [skill1, skill2, ...]}</li>
    <li><i>m1_researcher_skills{}</i> - {researcher: [skill1, skill2, ...]}</li>
    <li><i>m1_pseudo_researcher_skills{}</i> - {proposal: {researcher: [skill1, skill2, ...], ...}}</li>
</ol>

In [ ]:
# Start time
print("Start time:\t", datetime.datetime.now())

# import Ultra-Metric 
import metrics_scorer as metrics

m1_goodness_scores={}

# for each proposal
for proposal in m1_teaming:
    #initialize
    m1_goodness_scores[proposal]=[]
    
    # for each researcher
    for researcher in range(len(m1_teaming[proposal])):
        goodness_for_each_researcher=[m1_teaming[proposal][researcher][0],[]]
        
        # for each team
        for index in range(len(m1_teaming[proposal][researcher][1])):     # [0] - contains researcher's name, where [1] contains teams
            # initialize to team
            team=m1_teaming[proposal][researcher][1][index]
            
            # Apply Ultra-Metric (demand, team, researchers)
            temp_team_goodness=M1.apply_ultra_metric(m1_proposal_skills[proposal], team, m1_pseudo_researcher_skills[proposal])
            
            # save to scores
            goodness_for_each_researcher[1].append(temp_team_goodness)
    
        # save to overall dictionary
        m1_goodness_scores[proposal].append(goodness_for_each_researcher)
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:33:16.564025


In [ ]:
# export m1_goodness_scores
csv_m1_goodness_scores=[]
for i in m1_goodness_scores:
    for j in m1_goodness_scores[i]:
        for k in j[1]:
            csv_m1_goodness_scores.append([i, j[0], k])
            
csv_m1_goodness_scores=pd.DataFrame(csv_m1_goodness_scores, columns = ['nsf_proposal_links_v1', 'researcher_name', 'goodness'])
csv_m1_goodness_scores.to_csv(save_dir+'m1_goodness_scores.csv', encoding='utf-8')
del csv_m1_goodness_scores

### Step 6 - Final exports to CSV. <a id='method_import_6'></a> 

<b>Variables:</b>
<ol>
    <li>[DONE] <i>m1_teaming{}</i> - {proposal_link: [[researcher1, [team1, team2, ...]], [researcher2, [team1, team2,...]]]</li>
    <li>[DONE] <i>m1_proposal_skills{}</i> - {proposal_link: [skill1, skill2, ...]}</li>
    <li>[DONE] <i>m1_researcher_skills{}</i> - {researcher: [skill1, skill2, ...]}</li>
    <li>[DONE] <i>m1_goodness{}</i> - {proposal_link: [[researcher1, [goodness1, ...], [researcher2, [goodness1, ...]]</li>
    <li>Complete teaming data </li>
</ol>

In [ ]:
# data = [proposal_link, proposal_title, proposal_skills, researcher, team, goodness]

# Start time
print("Start time:\t", datetime.datetime.now())

# group together and export the teaming data
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers/"
csv_uc1_m1_teaming=[]
for i in m1_teaming:    # proposal
    for j in range(len(m1_teaming[i])):     # researcher
            
        # get title
        title_index=list(proposal_info['nsf_proposal_links_v1']).index(i)
        title=proposal_info['title'][title_index]

        # formatting variables (proposal year, proposal ID, proposal name + year)
        year=i.split("/")[4]
        proposal_id=i.split("/")[5]      # nsf#####
        csv_hyperlink_text=proposal_info['title'][title_index]+" ("+str(year)+")"     # Sample Proposal Name (2023)

        # sort teams in descending order (based on goodness scores)
        unsorted_teams=m1_teaming[i][j][1]
        unsorted_goodness=m1_goodness_scores[i][j][1]

        sorted_teams=[x for _,x in sorted(zip(unsorted_goodness, unsorted_teams), reverse=True)]
        sorted_goodness=sorted(unsorted_goodness, reverse=True) 
        
        # round goodness scores
        rounded_scores=[]
        for score in sorted_goodness:
            rounded_scores.append(round(score,4))
        
        # save
        csv_uc1_m1_teaming.append([proposal_id,
                                   year,
                                   i,
                                   #"=HYPERLINK(\""+i+"\", \""+csv_hyperlink_text+"\")",
                                   csv_hyperlink_text,
                                   m1_proposal_skills[i], 
                                   m1_teaming[i][j][0], 
                                   sorted_teams, 
                                   rounded_scores])

csv_uc1_m1_teaming=pd.DataFrame(csv_uc1_m1_teaming, columns = ['proposal_id', 'year', 'proposal_link', 'title', "skills", "researcher_name", "team", "goodness"])
csv_uc1_m1_teaming.to_csv(save_dir+'teaming_uc1_m1.csv', encoding='utf-8')
#del csv_uc1_m1_teaming

# End time
print("End time:\t", datetime.datetime.now())